# 银行股低 PB 策略回测

## 策略规则
- 选股：A 股银行板块，按市值排序取前 10
- 买入：市净率 (PB) < 0.6 时买入
- 仓位：首次买入按满足条件的股票数量等分 40%
- 补仓：相对上次买入价跌 10%，补 20%，按股票数量均分
- 卖出：有新股票要买时，卖涨幅最大的腾资金
- 补仓上限：无限，直到现金用完
- 调仓频率：每日检查
- 回测时间：2023-01-01 ~ 2026-08-22
- 手续费：佣金万3 + 印花税千0.5
- 基准：沪深300 + 银行ETF (512800)

In [1]:
import sys
sys.path.insert(0, '/Users/riceball/Documents/Projs/fund')

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/Users/riceball/Documents/Projs/fund/data')
FIN_DIR = DATA_DIR / 'financial'

BANK_STOCKS = {
    '601398': '工商银行', '601288': '农业银行', '601988': '中国银行',
    '601939': '建设银行', '601328': '交通银行', '601658': '邮储银行',
    '600036': '招商银行', '601166': '兴业银行', '600016': '民生银行',
    '600000': '浦发银行', '601818': '光大银行', '601998': '中信银行',
    '000001': '平安银行', '002142': '宁波银行', '601009': '南京银行',
    '601838': '成都银行', '600926': '杭州银行', '601077': '渝农商行',
    '601577': '长沙银行', '601169': '北京银行', '600919': '江苏银行',
    '601229': '上海银行', '601528': '瑞丰银行', '600908': '无锡银行',
    '601128': '常熟银行', '601860': '紫金银行', '601665': '苏农银行',
    '603323': '苏州银行', '002839': '张家港行', '002948': '郑州银行',
    '601963': '重庆银行', '600928': '西安银行', '001227': '兰州银行',
}
print(f'银行板块股票数量: {len(BANK_STOCKS)}')

银行板块股票数量: 33


In [2]:
financial_cache = {}

def load_financial_data(code):
    if code in financial_cache:
        return financial_cache[code]
    result = {'bvps': {}, 'shares': {}, 'total_equity': {}}
    fin_path = FIN_DIR / f'{code}_fin.csv'
    if fin_path.exists():
        df = pd.read_csv(fin_path, header=0)
        bvps_row = df[df.iloc[:, 1] == '每股净资产']
        if not bvps_row.empty:
            row = bvps_row.iloc[0]
            for col in df.columns[2:]:
                try:
                    rd = pd.to_datetime(str(col), format='%Y%m%d')
                    v = float(row[col]) if pd.notna(row[col]) else None
                    if v and v > 0:
                        result['bvps'][rd] = v
                except: pass
        equity_row = df[df.iloc[:, 1] == '股东权益合计(净资产)']
        if not equity_row.empty:
            row = equity_row.iloc[0]
            for col in df.columns[2:]:
                try:
                    rd = pd.to_datetime(str(col), format='%Y%m%d')
                    v = float(row[col]) if pd.notna(row[col]) else None
                    if v and v > 0:
                        result['total_equity'][rd] = v
                except: pass
    for rd in result['bvps']:
        if rd in result['total_equity']:
            b, e = result['bvps'][rd], result['total_equity'][rd]
            if b > 0 and e > 0:
                result['shares'][rd] = e / b
    financial_cache[code] = result
    return result

for code in BANK_STOCKS:
    load_financial_data(code)
print(f'已预加载 {len(financial_cache)} 只银行股财务数据')

已预加载 33 只银行股财务数据


In [3]:
all_market_data = {}
valid_banks = {}

for code, name in BANK_STOCKS.items():
    qfq = DATA_DIR / f'{code}_qfq.csv'
    if qfq.exists():
        df = pd.read_csv(qfq)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        if len(df) > 100:
            all_market_data[code] = df
            fd = financial_cache.get(code, {})
            if fd.get('bvps') and fd.get('shares'):
                valid_banks[code] = name

print(f'有效银行股数量: {len(valid_banks)}')
for c, n in sorted(valid_banks.items()):
    print(f'  {c} {n}')

有效银行股数量: 21
  000001 平安银行
  002142 宁波银行
  600016 民生银行
  600036 招商银行
  600919 江苏银行
  600926 杭州银行
  601009 南京银行
  601077 渝农商行
  601166 兴业银行
  601169 北京银行
  601229 上海银行
  601288 农业银行
  601328 交通银行
  601398 工商银行
  601658 邮储银行
  601818 光大银行
  601838 成都银行
  601939 建设银行
  601963 重庆银行
  601988 中国银行
  601998 中信银行


In [4]:
def _pit_value(data_dict, as_of):
    for rd in sorted(data_dict.keys(), reverse=True):
        m, y = rd.month, rd.year
        if m == 12: pub = pd.Timestamp(y+1, 4, 30)
        elif m == 3: pub = pd.Timestamp(y, 4, 30)
        elif m == 6: pub = pd.Timestamp(y, 8, 31)
        elif m == 9: pub = pd.Timestamp(y, 10, 31)
        else: continue
        if pub <= as_of:
            return data_dict[rd]
    return None

def screen_top10_banks(as_of):
    candidates = []
    for code, name in valid_banks.items():
        if code not in all_market_data: continue
        df = all_market_data[code]
        if as_of not in df.index: continue
        price = df.loc[as_of, 'close']
        if price <= 0: continue
        fd = financial_cache[code]
        bvps = _pit_value(fd['bvps'], as_of)
        shares = _pit_value(fd['shares'], as_of)
        if bvps and bvps > 0 and shares and shares > 0:
            pb = price / bvps
            mcap = shares * price
            candidates.append({'code': code, 'name': name, 'price': price, 'pb': pb, 'market_cap': mcap})
    candidates.sort(key=lambda x: x['market_cap'], reverse=True)
    top10 = candidates[:10]
    return [c for c in top10 if c['pb'] < 0.6]

test_date = pd.Timestamp('2024-06-15')
q = screen_top10_banks(test_date)
print(f'{test_date.date()} 满足条件: {len(q)} 只')
for s in q:
    print(f"  {s['code']} {s['name']}: PB={s['pb']:.3f}, 市值={s['market_cap']/1e8:.0f}亿")

2024-06-15 满足条件: 0 只


In [5]:
class BankPBBacktest:
    def __init__(self, initial_capital=1_000_000, commission=0.0003, stamp_tax=0.0005):
        self.initial_capital = initial_capital
        self.commission = commission
        self.stamp_tax = stamp_tax
        self.cash = initial_capital
        self.positions = {}
        self.nav_history = []
        self.trade_history = []

    def get_nav(self, as_of):
        total = self.cash
        for code, pos in self.positions.items():
            if code in all_market_data and as_of in all_market_data[code].index:
                total += pos['shares'] * all_market_data[code].loc[as_of, 'close']
        return total

    def buy(self, code, name, price, amount, as_of, reason):
        if amount <= 0 or amount > self.cash: return
        shares = int(amount / price / 100) * 100
        if shares <= 0: return
        actual_amount = shares * price
        self.cash -= (actual_amount + actual_amount * self.commission)
        if code in self.positions:
            pos = self.positions[code]
            ts = pos['shares'] + shares
            pos['avg_cost'] = (pos['avg_cost'] * pos['shares'] + price * shares) / ts
            pos['shares'] = ts
            pos['last_buy_price'] = price
        else:
            self.positions[code] = {'shares': shares, 'avg_cost': price, 'last_buy_price': price, 'name': name}
        self.trade_history.append({'date': as_of, 'code': code, 'name': name, 'action': 'BUY',
            'price': price, 'shares': shares, 'amount': actual_amount, 'reason': reason})

    def sell(self, code, price, as_of, reason):
        if code not in self.positions: return
        pos = self.positions[code]
        amt = pos['shares'] * price
        self.cash += (amt - amt * self.commission - amt * self.stamp_tax)
        self.trade_history.append({'date': as_of, 'code': code, 'name': pos['name'], 'action': 'SELL',
            'price': price, 'shares': pos['shares'], 'amount': amt, 'reason': reason})
        del self.positions[code]

    def run(self, start_date, end_date):
        start, end = pd.Timestamp(start_date), pd.Timestamp(end_date)
        trading_days = all_market_data[list(all_market_data.keys())[0]].index
        trading_days = trading_days[(trading_days >= start) & (trading_days <= end)]
        print(f'回测期间: {start_date} ~ {end_date}, 共 {len(trading_days)} 个交易日')

        for i, today in enumerate(trading_days):
            for code in list(self.positions.keys()):
                if code in all_market_data and today in all_market_data[code].index:
                    self.positions[code]['current_price'] = all_market_data[code].loc[today, 'close']

            qualified = screen_top10_banks(today)
            qualified_codes = {s['code'] for s in qualified}

            if qualified and self.cash < self.initial_capital * 0.1:
                max_gain, sell_code = -1, None
                for code, pos in self.positions.items():
                    if code not in qualified_codes and 'current_price' in pos:
                        g = (pos['current_price'] - pos['avg_cost']) / pos['avg_cost']
                        if g > max_gain:
                            max_gain, sell_code = g, code
                if sell_code and sell_code in all_market_data and today in all_market_data[sell_code].index:
                    self.sell(sell_code, all_market_data[sell_code].loc[today, 'close'], today, '换股')

            for code in list(self.positions.keys()):
                pos = self.positions[code]
                if 'current_price' in pos and pos['last_buy_price'] > 0:
                    drop = (pos['last_buy_price'] - pos['current_price']) / pos['last_buy_price']
                    if drop >= 0.1:
                        n = max(len(qualified), 1)
                        add_amt = self.initial_capital * 0.2 / n
                        if add_amt <= self.cash:
                            self.buy(code, pos['name'], pos['current_price'], add_amt, today, f'补仓跌{drop*100:.0f}%')

            if qualified:
                new = [s for s in qualified if s['code'] not in self.positions]
                if new and self.cash > 0:
                    per = self.initial_capital * 0.4 / len(qualified)
                    for s in new:
                        if s['code'] not in self.positions and per <= self.cash:
                            self.buy(s['code'], s['name'], s['price'], per, today, '首次建仓')

            nav = self.get_nav(today)
            self.nav_history.append({'date': today, 'nav': nav, 'cash': self.cash, 'n_positions': len(self.positions)})

            if (i + 1) % 100 == 0:
                print(f'  进度: {i+1}/{len(trading_days)}, 净值: {nav:,.0f}, 持仓: {len(self.positions)}')

        print(f'回测完成! 最终净值: {self.nav_history[-1]["nav"]:,.0f}, 总交易: {len(self.trade_history)} 笔')

In [6]:
backtest = BankPBBacktest(initial_capital=1_000_000)
backtest.run('2023-01-01', '2026-08-22')

回测期间: 2023-01-01 ~ 2026-08-22, 共 876 个交易日
  进度: 100/876, 净值: 1,059,725, 持仓: 9


  进度: 200/876, 净值: 1,059,944, 持仓: 9


  进度: 300/876, 净值: 1,118,382, 持仓: 10


  进度: 400/876, 净值: 1,249,004, 持仓: 10


  进度: 500/876, 净值: 1,304,256, 持仓: 10
  进度: 600/876, 净值: 1,425,623, 持仓: 10


  进度: 700/876, 净值: 1,465,944, 持仓: 10
  进度: 800/876, 净值: 1,412,553, 持仓: 10


回测完成! 最终净值: 1,415,336, 总交易: 14 笔


In [7]:
def load_benchmark(code, start, end):
    # Try qfq first, then market
    for suffix in ["_qfq.csv", "_market.csv"]:
        p = DATA_DIR / f"{code}{suffix}"
        if p.exists():
            df = pd.read_csv(p)
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)
            return df[(df.index >= start) & (df.index <= end)]
    return None

hs300 = load_benchmark("000300", "2023-01-01", "2026-08-22")
bank_etf = load_benchmark("512800", "2023-01-01", "2026-08-22")
print(f"沪深300: {len(hs300) if hs300 is not None else 0} 行, 银行ETF: {len(bank_etf) if bank_etf is not None else 0} 行")

沪深300: 640 行, 银行ETF: 0 行


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

nav_df = pd.DataFrame(backtest.nav_history).set_index('date')
nav_df['return'] = nav_df['nav'] / nav_df['nav'].iloc[0] - 1

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax1 = axes[0, 0]
ax1.plot(nav_df.index, nav_df['nav'] / 1_000_000, label='银行低PB策略', linewidth=2)
if hs300 is not None:
    hs300_ret = hs300['close'] / hs300['close'].iloc[0]
    ax1.plot(hs300.index, hs300_ret, label='沪深300', alpha=0.7)
if bank_etf is not None:
    betf_ret = bank_etf['close'] / bank_etf['close'].iloc[0]
    ax1.plot(bank_etf.index, betf_ret, label='银行ETF', alpha=0.7)
ax1.set_title('净值曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
dd = (nav_df['nav'] - nav_df['nav'].cummax()) / nav_df['nav'].cummax()
ax2.fill_between(dd.index, dd, 0, alpha=0.5, color='red')
ax2.set_title('回撤')
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3.bar(nav_df.index, nav_df['n_positions'], alpha=0.7)
ax3.set_title('持仓数量')
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
ax4.plot(nav_df.index, nav_df['cash'] / nav_df['nav'], alpha=0.7)
ax4.set_title('现金比例')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/riceball/Documents/Projs/fund/reports/generated/bank_pb_strategy.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存')

图表已保存


In [9]:
def calc_perf(nav_series, name):
    rets = nav_series.pct_change().dropna()
    total = nav_series.iloc[-1] / nav_series.iloc[0] - 1
    n_yr = (nav_series.index[-1] - nav_series.index[0]).days / 365.25
    ann_ret = (1 + total) ** (1 / n_yr) - 1
    ann_vol = rets.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    dd = (nav_series - nav_series.cummax()) / nav_series.cummax()
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    return {'策略': name, '累计收益': f'{total*100:.2f}%', '年化收益': f'{ann_ret*100:.2f}%',
            '年化波动': f'{ann_vol*100:.2f}%', 'Sharpe': f'{sharpe:.2f}',
            '最大回撤': f'{max_dd*100:.2f}%', 'Calmar': f'{calmar:.2f}'}

results = [calc_perf(nav_df['nav'], '银行低PB策略')]
if hs300 is not None: results.append(calc_perf(hs300['close'], '沪深300'))
if bank_etf is not None: results.append(calc_perf(bank_etf['close'], '银行ETF'))
perf_df = pd.DataFrame(results)
print(perf_df.to_string(index=False))

     策略   累计收益   年化收益   年化波动 Sharpe    最大回撤 Calmar
银行低PB策略 41.55% 10.10% 10.89%   0.93  -9.43%   1.07
  沪深300 34.62% 11.90% 18.84%   0.63 -15.66%   0.76


In [10]:
trades_df = pd.DataFrame(backtest.trade_history)
print(f'交易记录 (共 {len(trades_df)} 笔):')
if not trades_df.empty:
    print(trades_df[['date','code','name','action','price','shares','amount','reason']].to_string())

交易记录 (共 14 笔):
         date    code  name action  price  shares   amount  reason
0  2023-01-03  601398  工商银行    BUY   3.54   14100  49914.0    首次建仓
1  2023-01-03  601939  建设银行    BUY   4.60   10800  49680.0    首次建仓
2  2023-01-03  601288  农业银行    BUY   2.39   20900  49951.0    首次建仓
3  2023-01-03  601988  中国银行    BUY   2.61   19100  49851.0    首次建仓
4  2023-01-03  601658  邮储银行    BUY   3.83   13000  49790.0    首次建仓
5  2023-01-03  601328  交通银行    BUY   3.83   13000  49790.0    首次建仓
6  2023-01-03  601166  兴业银行    BUY  14.02    3500  49070.0    首次建仓
7  2023-01-03  601998  中信银行    BUY   4.08   12200  49776.0    首次建仓
8  2023-03-09  000001  平安银行    BUY  10.82    4100  44362.0    首次建仓
9  2023-05-30  000001  平安银行    BUY   9.73    2200  21406.0  补仓跌10%
10 2023-10-25  000001  平安银行    BUY   8.72    2500  21800.0  补仓跌10%
11 2023-11-14  601166  兴业银行    BUY  12.49    1700  21233.0  补仓跌11%
12 2023-12-08  000001  平安银行    BUY   7.81    2800  21868.0  补仓跌10%
13 2023-12-08  601818  光大银行    BUY   2.48   179

In [11]:
import os
os.makedirs('/Users/riceball/Documents/Projs/fund/reports/generated', exist_ok=True)
nav_df.to_csv('/Users/riceball/Documents/Projs/fund/reports/generated/bank_pb_nav.csv')
if not trades_df.empty:
    trades_df.to_csv('/Users/riceball/Documents/Projs/fund/reports/generated/bank_pb_trades.csv', index=False)
perf_df.to_csv('/Users/riceball/Documents/Projs/fund/reports/generated/bank_pb_performance.csv', index=False)
print('结果已保存')

结果已保存
